In [ ]:
from NumOpt import Opti,ca 
import numpy as np 

opti=Opti()

x=opti.variable(init_guess=np.array([0.0]))
p=opti.parameter(value=0.5)
f=(x-p)**2
opti.minimize(f)
opti.ipopt_solver()
nlp:ca.Function=opti.to_function("nlp",[x,p],[x],["x0","p"],["x_star"])

C=ca.CodeGenerator("func.c")
C.add(nlp)
C.generate()

print(ca.GlobalOptions.getCasadiPath())
! gcc -O3 -fPIC -shared -o func.dll ./func.c -I D:/micromamba/envs/py12/Lib/site-packages/casadi/include/ -L D:/micromamba/envs/py12/Lib/site-packages/casadi -lipopt -lm
# ! cl /LD /O2 func.c /I D:/micromamba/envs/py12/Lib/site-packages/casadi/include /Fe:func.dll /link /LIBPATH:D:/micromamba/envs/py12/Lib/site-packages/casadi ipopt.lib

In [ ]:
import casadi as ca
import numpy as np
import subprocess

x = ca.MX.sym("x")
y = ca.MX.sym("y")

mingw_jit_options = {
    "jit": True,
    "compiler": "shell",
    "jit_options": {
        "compiler": "gcc",  # Force GCC instead of cl.exe
        "linker": "gcc",  # Force GCC for linking
        "compiler_setup": "-fPIC -c",  # GNU compilation flags
        "linker_setup": "-shared",  # GNU linking flags
        "compiler_output_flag": "-o ",
        "linker_output_flag": "-o ",
        "flags": [
            "-O3",
            "-lm"
            "-march=native",
            "-ffast-math",
            # "-flto",
            # "-fopenmp",
        ],  # Optimize output code
        "verbose": True,
    },
}

funcb = ca.Function("funcb", [x], [ca.sin(x)], ["x"], ["o1"], mingw_jit_options)

import gc
del funcb
gc.collect()